# BusinessGPT Reward Model: rubert-base pairwise ranker

Trains a small Russian BERT (`DeepPavlov/rubert-base-cased`) as a pairwise reward model on labeled preference pairs (`eval/preference_pairs_v16_multi.jsonl`, fresh v16 pairs).

**Pipeline position:**

1. SFT (`training.ipynb`) → v16 SFT
2. ORPO (`orpo.ipynb`) → v16-orpo
3. **This notebook** → reward model on HF (`vXofi/businessgpt-reward-rubert`)
4. Best-of-N inference (`businessgpt_bench.ipynb` + `eval_only.ipynb` extensions): generate 8 candidates, score with RM, return best

**Why rubert-base (180MB) not sbert_large (700MB):**
- RM runs at inference time alongside the 9B GGUF (6.4GB) — production budget is 12GB CPU RAM. 700MB of RM eats the headroom.
- 1450 training pairs is too small for sbert-large to outperform — it'd overfit.

**Gate**: held-out pairwise accuracy ≥ 0.75. If 0.70-0.75 → escalate to sbert-large. If <0.70 → labels are noisy, skip best-of-N integration.

**Requirements**: Kaggle GPU T4 (single GPU is enough; rubert-base is trivial).

## 0. Install Dependencies

In [ ]:
%%capture
%pip install -U transformers "trl==0.23.1" datasets accelerate huggingface_hub bitsandbytes

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

## 1. Config

In [ ]:
# Edit these per run.
RM_BASE      = "DeepPavlov/rubert-base-cased"
RM_REPO      = "vXofi/businessgpt-reward-rubert"
PAIRS_FILE   = "preference_pairs_v16_multi.jsonl"

MAX_LEN      = 512
HELD_OUT_N   = 50    # pairs reserved for the final accuracy check (never seen in train/val)
VAL_FRAC     = 0.20  # of the remaining ~1400, 20% goes to val
SEED         = 42

SAVE_DIR     = "rm_rubert_v16"

print(f"RM base: {RM_BASE}")
print(f"Output:  {RM_REPO}")
print(f"Pairs:   {PAIRS_FILE}")

## 2. Load preference pairs

In [ ]:
import json
import os
import glob as _glob

_candidates = [
    f"/kaggle/input/businessgpt-eval/{PAIRS_FILE}",
    f"/kaggle/input/businessgpt-eval/eval/{PAIRS_FILE}",
    f"eval/{PAIRS_FILE}",
    PAIRS_FILE,
]
_pairs_path = next((p for p in _candidates if os.path.isfile(p)), None)
if _pairs_path is None:
    _matches = _glob.glob(f"/kaggle/input/**/{PAIRS_FILE}", recursive=True)
    if _matches:
        _pairs_path = _matches[0]
if _pairs_path is None:
    raise FileNotFoundError(
        f"{PAIRS_FILE} not found. Attach the businessgpt-eval Kaggle dataset."
    )

with open(_pairs_path, encoding="utf-8") as f:
    pairs = [json.loads(line) for line in f if line.strip()]
print(f"Loaded {len(pairs)} pairs from {_pairs_path}")

from collections import Counter
print(f"  by category: {Counter(p['category'] for p in pairs)}")

## 3. Serialize each pair as full chat template

Same `<|im_start|>{role}\n{content}<|im_end|>` format that the SFT model receives — so the RM scores inputs in the same shape it'll see at inference time. The RM doesn't need to *parse* the special tokens (rubert will subword-split them); it just learns to correlate the surface form with chosen/rejected labels.

In [ ]:
def serialize(prompt_messages, response_text):
    parts = [
        f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>" for m in prompt_messages
    ]
    parts.append(f"<|im_start|>assistant\n{response_text}<|im_end|>")
    return "\n".join(parts)


records = []
for p in pairs:
    records.append({
        "chosen":    serialize(p["prompt"], p["chosen"][-1]["content"]),
        "rejected":  serialize(p["prompt"], p["rejected"][-1]["content"]),
        "prompt_id": p["prompt_id"],
        "category":  p["category"],
    })

# Print one example for sanity
print("--- Sample chosen ---")
print(records[0]["chosen"][:400])
print()
print("--- Sample rejected ---")
print(records[0]["rejected"][:400])

## 4. Train / val / held-out split

In [ ]:
import random as _random

_random.seed(SEED)
_random.shuffle(records)

held_out = records[:HELD_OUT_N]
remaining = records[HELD_OUT_N:]
val_size = int(VAL_FRAC * len(remaining))
val_records = remaining[:val_size]
train_records = remaining[val_size:]

print(f"Train:    {len(train_records)}")
print(f"Val:      {len(val_records)}")
print(f"Held-out: {len(held_out)}  (reserved for final accuracy gate)")

## 5. Load model + tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(RM_BASE)
# Long chat contexts get truncated; we want to PRESERVE the recent response
# (which is what the RM judges) and drop the front of the conversation.
tokenizer.truncation_side = "left"
print(f"Tokenizer: {RM_BASE}, truncation_side=left, max_len={MAX_LEN}")

model = AutoModelForSequenceClassification.from_pretrained(
    RM_BASE,
    num_labels=1,
    trust_remote_code=True,
)
print(f"Model: {model.__class__.__name__}, params={sum(p.numel() for p in model.parameters()):,}")

### Truncation sanity check

Verify the truncation actually preserves the `<|im_start|>assistant\n...<|im_end|>` suffix — losing the response itself would defeat the reward signal.

In [ ]:
_sample = max(records, key=lambda r: len(r["chosen"]))
print(f"Longest chosen serialization: {len(_sample['chosen'])} chars")
_enc = tokenizer(_sample["chosen"], truncation=True, max_length=MAX_LEN, return_tensors="pt")
_decoded = tokenizer.decode(_enc["input_ids"][0])
# Check assistant block is still present after truncation
_has_assistant = "assistant" in _decoded
print(f"Token count after truncation: {_enc['input_ids'].shape[1]}")
print(f"Assistant block preserved: {_has_assistant}")
if not _has_assistant:
    print("⚠️  Assistant block dropped! Raise MAX_LEN or shorten contexts.")
print(f"Last 200 chars of decoded:\n{_decoded[-200:]!r}")

## 6. Train

In [ ]:
from trl import RewardTrainer, RewardConfig
from datasets import Dataset

train_ds = Dataset.from_list(train_records)
val_ds   = Dataset.from_list(val_records)

NUM_EPOCHS = 4
BATCH_SIZE = 8
LR         = 2e-5
STEPS_PER_EPOCH = max(1, len(train_ds) // BATCH_SIZE)

# RewardTrainer handles the pairwise loss internally: it scores chosen and rejected
# separately, applies -logsigmoid(score_chosen - score_rejected), backprops once.
rm_args = RewardConfig(
    output_dir="rm_outputs",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_steps=max(1, int(STEPS_PER_EPOCH * NUM_EPOCHS * 0.1)),
    lr_scheduler_type="cosine",
    max_length=MAX_LEN,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
    greater_is_better=True,
    fp16=True,
    seed=SEED,
    report_to="none",
)

# TRL 0.23 expects this dict on HF model classes; some Transformers models lack it.
if not hasattr(model, "warnings_issued"):
    model.warnings_issued = {}

trainer = RewardTrainer(
    model=model,
    args=rm_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)

print(f"Training config:")
print(f"  Examples: {len(train_ds)} train / {len(val_ds)} val")
print(f"  Epochs={NUM_EPOCHS}, batch={BATCH_SIZE}, lr={LR}, max_len={MAX_LEN}")
trainer.train()
print(f"\nFinal train metrics:")
for k, v in trainer.state.log_history[-1].items():
    print(f"  {k}: {v}")

## 7. Held-out accuracy gate

Strict threshold: **≥ 0.75 pairwise accuracy** on the 50 held-out pairs.

- < 0.70: labels too noisy → don't ship best-of-N, document failure.
- 0.70-0.75: escalate to `sbert_large_nlu_ru`, re-train.
- ≥ 0.75: ship it.

In [ ]:
import torch

device = next(model.parameters()).device
model.eval()

@torch.no_grad()
def score(text):
    enc = tokenizer(text, truncation=True, max_length=MAX_LEN,
                    return_tensors="pt").to(device)
    return model(**enc).logits.squeeze().item()


correct = 0
margins = []
for r in held_out:
    s_chosen   = score(r["chosen"])
    s_rejected = score(r["rejected"])
    if s_chosen > s_rejected:
        correct += 1
    margins.append(s_chosen - s_rejected)

acc = correct / len(held_out)
mean_margin = sum(margins) / len(margins)
median_margin = sorted(margins)[len(margins) // 2]

print(f"Held-out pairwise accuracy: {correct}/{len(held_out)} = {acc:.3f}")
print(f"Margin (score_chosen - score_rejected): mean={mean_margin:.3f}, median={median_margin:.3f}")
print()
if acc >= 0.75:
    print("✓ GATE PASSED (≥0.75) — ship the model, wire into best-of-N inference")
    GATE_PASSED = True
elif acc >= 0.70:
    print("⚠ Borderline (0.70-0.75): consider escalating to sbert_large_nlu_ru and re-running")
    GATE_PASSED = False
else:
    print("✗ GATE FAILED (<0.70): labels too noisy. Don't ship best-of-N. Investigate label consistency.")
    GATE_PASSED = False

## 8. Save + push (only if gate passed)

In [ ]:
if not GATE_PASSED:
    print("Gate not passed — skipping push to avoid shipping a weak RM. Re-run training with sbert-large or investigate labels first.")
else:
    import os
    os.makedirs(SAVE_DIR, exist_ok=True)
    model.save_pretrained(SAVE_DIR)
    tokenizer.save_pretrained(SAVE_DIR)

    # Write a README with the accuracy metrics inline.
    readme = f"""---
library_name: transformers
tags: [reward-model, russian, rubert]
base_model: {RM_BASE}
---

# BusinessGPT Reward Model (rubert-base)

Trained on {len(train_records)} preference pairs from BusinessGPT v16 multi-candidate labeling.

## Eval

- Held-out pairwise accuracy ({len(held_out)} pairs): **{acc:.3f}**
- Margin (chosen - rejected): mean={mean_margin:.3f}, median={median_margin:.3f}

## Use

```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tok = AutoTokenizer.from_pretrained("{RM_REPO}")
tok.truncation_side = "left"
mdl = AutoModelForSequenceClassification.from_pretrained("{RM_REPO}")

# Score a single (prompt, response) pair:
parts = [f"<|im_start|>{{m['role']}}\\n{{m['content']}}<|im_end|>" for m in prompt_messages]
parts.append(f"<|im_start|>assistant\\n{{response}}<|im_end|>")
text = "\\n".join(parts)
enc = tok(text, truncation=True, max_length=512, return_tensors="pt")
score = mdl(**enc).logits.squeeze().item()
```

Use for best-of-N re-ranking at inference time.
"""
    with open(f"{SAVE_DIR}/README.md", "w", encoding="utf-8") as f:
        f.write(readme)

    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(RM_REPO, exist_ok=True)
    api.upload_folder(
        folder_path=SAVE_DIR,
        repo_id=RM_REPO,
        commit_message=f"BusinessGPT reward model (acc={acc:.3f}, n_train={len(train_records)})",
    )
    print(f"Pushed to https://huggingface.co/{RM_REPO}")